# All sites plot Land Exposure compared to Low Tide

For the selected date range, process Sentinel-2 scenes with less than 10% catalogue cloud cover at all configured sites. For each scene, record the nearest predicted low-tide time, height, and signed time offset; estimate whole-site dry area from SCL vegetation (class 4) and not-vegetated (class 5) pixels. Save per-site CSVs, RGB/SCL galleries, and dry-area-versus-low-tide scatter plots in each site's output folder.

In [2]:
import base64
import datetime
import html as html_module
import io
import os
import sys
import pathlib

import geopandas
import matplotlib.pyplot as plt
import numpy
import pandas
import plotly.express as px
import rasterio
import rasterio.features
import xarray
from IPython.display import FileLink, display

module_path = pathlib.Path.cwd().parent.parent / 'scripts'
import sys
if str(module_path) not in sys.path:
    sys.path.append(str(module_path))
import utils
import sentinel2
import plotting

%load_ext autoreload
%autoreload 2

## Date-Range Site and Cloud Settings

Set the start and end years and cloud-cover cutoff below. All matching scenes are retained; low-tide timing is recorded for comparison, not used to exclude scenes.

In [3]:
data_path = utils.get_data_path()

In [4]:
all_training_sites =  ["Ihutai", "CatlinsLake", "CatlinsRiverMouth", "Childrens", "Duvauchelle", "Robinsons", "Takamatua", "Purau",
              "IveyBay_Nov25", "IveyBay_Feb26", "LeftBank_Nov25", "LeftBank_Feb26", "Paremata_Nov25", "Paremata_Feb26",
              "Paremata_Feb25", "ThePoint_Nov25", "ThePoint_Feb26", "Takapuwahia_Nov25", "Takapuwahia_Feb26", "IveyBay_ThePoint_LeftBank_Oct24"]
sites = all_training_sites

years = [2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]

max_cloud_cover = 5

In [5]:
_, satellite_path = utils.get_prediction_path(
    sample_method="sampling_2", method_2_threshold=0.5,
    model_max_cloud_cover=max_cloud_cover,
    predict_max_cloud_cover=max_cloud_cover,
    satellite_only=True) 

mndwi_file = satellite_path / "mean_mndwi.csv"
tide_time_file = satellite_path / "tide_time.csv"
mndwi_output_path = satellite_path / "rbg_thumbnails_mndwi_sorted"
mndwi_output_path = satellite_path / "rbg_thumbnails_tide_sorted"
output_path.mkdir(exist_ok=True, parents=True)

NameError: name 'output_path' is not defined

# Thumbnails for 5% cloud sorted by tide

In [ ]:
tide_times_all_sites = []

for site in sites:
    tide_time_series = []

    site_polygon_file = utils.get_site_polygon_path(site)
    site_polygon = geopandas.read_file(site_polygon_file)
    centroid_nzgd49 = site_polygon.to_crs(utils.CRS_NZGD49).geometry.union_all().centroid
    latitude_nzgd49, longitude_nzgd49 = float(centroid_nzgd49.y), float(centroid_nzgd49.x)
    
    for year in years:
        print(f"{site}: year {year}")
    
        satellite_file = satellite_path / f"{site}_{year}_sentinel-2.nc"
        if not satellite_file.exists():
            print(f"WARNING no {satellite_file.name}.")
            continue

        # load in satellite file
        satellite_data = utils.load_satellite(filename=satellite_file, chunks=None)
        # calculate distances from low tide
        timestamps = satellite_data["time"].dt.strftime("%Y-%m-%dT%H:%M:%S.%f").values
        tide_info = []
        for timestamp in timestamps:
            tide_dict = sentinel2._get_low_tide_info_for_datetime(timestamp, latitude_nzgd49, longitude_nzgd49)
            tide_dict["time"] = timestamp
            tide_info.append(tide_dict)
        tide_time_series.append(pandas.DataFrame(tide_info))
    tide_time_series = pandas.concat(tide_time_series)
    tide_times_all_sites.append(tide_time_series)

# for plotting the gallery / distributions
tide_times_all_sites = pandas.concat(tide_times_all_sites, keys=sites, names=["site", "date id"])
tide_times_all_sites.to_csv(tide_time_file)

In [ ]:
tide_times_all_sites = pandas.read_csv(tide_time_file)
tide_times_all_sites["time"] = pandas.to_datetime(tide_times_all_sites["time"], utc=True)
tide_times_all_sites["minutes_from_low_tide"] = pandas.to_timedelta(tide_times_all_sites["time_from_low_tide"]).dt.total_seconds()/60
output_folder = satellite_path /  "rbg_thumbnails_tide_sorted"
output_folder.mkdir(exist_ok=True)

for site in sites:
    print(f"{site}")
    gallery_file = output_folder / f"{site}.html"
    if gallery_file.exists():
        continue
    plotting.create_rgb_gallery(site_name=site,
                                sort_value=tide_times_all_sites, 
                                sort_value_label="minutes_from_low_tide",
                                gallery_file=gallery_file, 
                                years=years,
                                max_cloud_cover=max_cloud_cover,
                                satellite_path=satellite_path)

In [ ]:
mean_mndwi_all_sites = []

for site in sites:
    mean_mndwi_series = []
    for year in years:
        print(f"{site}: year {year}")
    
        satellite_file = satellite_path / f"{site}_{year}_sentinel-2.nc"
        if not satellite_file.exists():
            print(f"WARNING no {satellite_file.name}.")
            continue

        # load in satellite file
        satellite_data = utils.load_satellite(filename=satellite_file, chunks=None)
        # calculate MNDWI layer
        mean_mndwi = sentinel2._mean_mndwi_by_date(satellite_data)
        mean_mndwi_series.append(
            mean_mndwi.compute().to_pandas()
        )
    mean_mndwi_series = pandas.concat(mean_mndwi_series)
    mean_mndwi_all_sites.append(mean_mndwi_series)

# for plotting the gallery / distributions
mean_mndwi_all_sites = pandas.concat(mean_mndwi_all_sites, keys=sites, names=["site", "time"]).rename("mean_mndwi")
mean_mndwi_all_sites.to_csv(mean_mndwi_file)

In [ ]:
mean_mndwi = pandas.read_csv(mean_mndwi_file)
output_folder = satellite_path /  "rbg_thumbnails_mndwi_sorted"
output_folder.mkdir(exist_ok=True)

for site in sites:
    print(f"{site}")
    gallery_file = output_folder / f"{site}.html"
    if gallery_file.exists():
        continue
    plotting.create_rgb_gallery(site_name=site,
                                sort_value=mean_mndwi, 
                                sort_value_label="Mean MNDWI",
                                gallery_file=gallery_file, 
                                years=years,
                                max_cloud_cover=baseline_max_cloud_cover,
                                satellite_path=satellite_path)